In [ ]:
import pandas as pd
import numpy as np

# --- Load your PO history file ---
# Update the path/filename to match yours
df = pd.read_csv('spend_clean.csv')  # or read_excel if .xlsx

# Ensure RelExtCost is numeric
df['RelExtCost'] = pd.to_numeric(df['RelExtCost'], errors='coerce')

# --- ABC Analysis: Annual spend per part ---
abc = (
    df.groupby('PartNum')['RelExtCost']
    .sum()
    .reset_index()
    .rename(columns={'RelExtCost': 'TotalSpend'})
    .sort_values('TotalSpend', ascending=False)
)

# Cumulative spend %
abc['CumSpend'] = abc['TotalSpend'].cumsum()
abc['CumSpend_Pct'] = abc['CumSpend'] / abc['TotalSpend'].sum() * 100

# Assign A / B / C
def abc_label(pct):
    if pct <= 80:
        return 'A'
    elif pct <= 95:
        return 'B'
    else:
        return 'C'

abc['ABC'] = abc['CumSpend_Pct'].apply(abc_label)

# --- Summary ---
summary = abc.groupby('ABC').agg(
    PartCount=('PartNum', 'count'),
    TotalSpend=('TotalSpend', 'sum')
).reset_index()
summary['Spend_Pct'] = (summary['TotalSpend'] / summary['TotalSpend'].sum() * 100).round(1)
summary['Part_Pct'] = (summary['PartCount'] / summary['PartCount'].sum() * 100).round(1)

print(summary.to_string(index=False))
print(f"\nTotal parts: {len(abc)}")

In [ ]:
# --- XYZ Analysis: Demand variability per part ---
# Using RelQty (order quantity) as the demand signal, grouped by month

df['OrderDate'] = pd.to_datetime(df['OrderDate'])
df['YearMonth'] = df['OrderDate'].dt.to_period('M')

# Monthly demand per part
monthly = (
    df.groupby(['PartNum', 'YearMonth'])['RelQty']
    .sum()
    .reset_index()
)

# Coefficient of Variation (CV) = std / mean
xyz = (
    monthly.groupby('PartNum')['RelQty']
    .agg(['mean', 'std', 'count'])
    .reset_index()
)
xyz.columns = ['PartNum', 'MeanQty', 'StdQty', 'MonthCount']
xyz['CV'] = xyz['StdQty'] / xyz['MeanQty']

# Only classify parts with enough history (at least 3 months)
xyz = xyz[xyz['MonthCount'] >= 2]

# Assign X / Y / Z
def xyz_label(cv):
    if cv <= 0.5:
        return 'X'
    elif cv <= 1.0:
        return 'Y'
    else:
        return 'Z'

xyz['XYZ'] = xyz['CV'].apply(xyz_label)

# --- Summary ---
xyz_summary = xyz.groupby('XYZ').agg(
    PartCount=('PartNum', 'count')
).reset_index()
xyz_summary['Part_Pct'] = (xyz_summary['PartCount'] / xyz_summary['PartCount'].sum() * 100).round(1)

print(xyz_summary.to_string(index=False))
print(f"\nParts with insufficient history (excluded): {len(df['PartNum'].unique()) - len(xyz)}")

In [ ]:
# --- Check excluded parts ---
all_parts = df['PartNum'].unique()
classified_parts = xyz['PartNum'].unique()
excluded = df[~df['PartNum'].isin(classified_parts)]

print("Excluded parts breakdown:")
print(excluded.groupby('PartNum')['OrderDate'].count().describe())
print(f"\nSample of excluded parts (low order count):")
print(
    excluded.groupby('PartNum')['OrderDate']
    .count()
    .reset_index()
    .rename(columns={'OrderDate': 'OrderCount'})
    .sort_values('OrderCount', ascending=False)
    .head(10)
)

In [ ]:
# --- Combine ABC + XYZ ---

# Label excluded parts as Z
all_parts = pd.DataFrame(df['PartNum'].unique(), columns=['PartNum'])
xyz_full = all_parts.merge(xyz[['PartNum', 'XYZ']], on='PartNum', how='left')
xyz_full['XYZ'] = xyz_full['XYZ'].fillna('Z')

# Merge with ABC
matrix = abc.merge(xyz_full, on='PartNum', how='left')
matrix['ABCXYZ'] = matrix['ABC'] + matrix['XYZ']

# --- Matrix summary ---
summary_matrix = matrix.groupby(['ABC', 'XYZ']).agg(
    PartCount=('PartNum', 'count'),
    TotalSpend=('TotalSpend', 'sum')
).reset_index()
summary_matrix['Spend_Pct'] = (summary_matrix['TotalSpend'] / summary_matrix['TotalSpend'].sum() * 100).round(1)

print(summary_matrix.to_string(index=False))

# Save for next phase
matrix.to_csv('abc_xyz_matrix.csv', index=False)
print("\nSaved: abc_xyz_matrix.csv")

In [ ]:
# =============================================================================
# PHASE 3: ABC/XYZ ANALYSIS
# =============================================================================
# Goal: Classify all parts by business value (ABC) and demand variability (XYZ)
# to determine the appropriate inventory strategy for each group.
#
# Data source: spend_clean.csv (output from Phase 1 data cleaning)
#
# --- ABC ANALYSIS --- = how much we spend on each part
# Parts ranked by total spend (RelExtCost) and assigned to tiers:
#   A = top 80% of cumulative spend
#   B = next 15% (80–95%)
#   C = bottom 5% (95–100%)
#
# Results (15,992 total parts):
#   A:  862 parts  |  5.4% of catalog  |  80% of spend
#   B: 2,045 parts | 12.8% of catalog  |  15% of spend
#   C: 13,085 parts | 81.8% of catalog  |   5% of spend
#
# Classic Pareto distribution confirmed — a small number of parts
# drive the vast majority of spend.
#
# --- XYZ ANALYSIS --- = how predictable is demand for each part
# Parts classified by Coefficient of Variation (CV) of monthly order quantity:
#   X = CV <= 0.5  (stable demand)
#   Y = CV <= 1.0  (moderate variability)
#   Z = CV >  1.0  (highly unpredictable)
#
# Parts with fewer than 2 months of order history (9,415 parts) were
# labeled Z by default — single-occurrence purchases with no demand pattern.
#
# Results (6,577 classified + 9,415 defaulted to Z):
#   X: 3,921 parts | 59.6% of classified
#   Y: 2,217 parts | 33.7% of classified
#   Z:   439 parts |  6.7% of classified (+ 9,415 defaulted)
#
# --- ABC/XYZ MATRIX --- = which parts deserve careful inventory management, and which don't
# Combined classification results:
#
#   Segment | Parts |  Spend%  | Implication
#   --------|-------|----------|--------------------------------------------
#   AX      |   350 |  29.5%   | High value, predictable → tight min/max
#   AY      |   309 |  33.3%   | High value, variable → safety stock needed
#   AZ      |   203 |  17.2%   | High value, unpredictable → demand-driven
#   BX      |   840 |   6.4%   | Moderate value, stable → standard reorder
#   BY      |   627 |   5.0%   | Moderate value, variable → review buffers
#   BZ      |   578 |   3.6%   | Moderate value, unpredictable → monitor
#   CX      | 2,731 |   1.6%   | Low value, stable → simple min/max or kanban
#   CY      | 1,281 |   0.9%   | Low value, variable → order as needed
#   CZ      | 9,073 |   2.5%   | Low value, unpredictable → no-stock/order only
#
# Key finding: AX + AY + AZ = 862 parts = 80% of total spend.
# Phase 4 safety stock modeling will focus on A parts first.
#
# Output saved: abc_xyz_matrix.csv
# =============================================================================